**Imports**

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path

**Datasets**

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


raw data

In [27]:
article_dir = Path('/content/drive/MyDrive/article10')
processed_dir = article_dir / 'data_processed'
processed_dir.mkdir(exist_ok=True)

fred_md_raw = pd.read_csv(
    article_dir / '2018-12.csv',
    header=None
)

fred_columns = fred_md_raw.iloc[0].tolist()

fred_transform_codes = pd.Series(
    fred_md_raw.iloc[1].tolist(),
    index=fred_columns
)

fred_md = fred_md_raw.iloc[2:].copy()
fred_md.columns = fred_columns
fred_md = fred_md.reset_index(drop=True)

liu_wu_raw = pd.read_excel(
    article_dir / 'LW_monthly.xlsx',
    header=8
)

liu_wu = liu_wu_raw.copy()
liu_wu = liu_wu.rename(
    columns={liu_wu.columns[0]: 'date'}
)

liu_wu.columns = [
    str(col).strip().replace(' ', '_')
    for col in liu_wu.columns
]

processed data

In [28]:
liu_wu['date'] = pd.PeriodIndex(
    liu_wu['date'].astype(str),
    freq='M'
)

liu_wu = liu_wu[
    (liu_wu['date'] >= pd.Period('1971-08', freq='M')) &
    (liu_wu['date'] <= pd.Period('2018-12', freq='M'))
].copy()

liu_wu = liu_wu.set_index('date')

yield_columns = liu_wu.columns

liu_wu[yield_columns] = liu_wu[yield_columns].apply(
    pd.to_numeric,
    errors='coerce'
)

Forward rate for the period from year n−1 to year n:


In [29]:
forward_rates = pd.DataFrame(index=liu_wu.index)
forward_rates['short_rate'] = liu_wu['12_m']
for n in range(2, 11):
    current_yield = liu_wu[f'{12*n}_m']
    previous_yield = liu_wu[f'{12*(n-1)}_m']

    forward_rates[f'fwd_{n}y'] = (
        n * current_yield - (n - 1) * previous_yield
    )

12-month excess returns:

In [30]:
target_maturities = [2, 3, 4, 5, 7, 10]

excess_returns = pd.DataFrame(index=liu_wu.index)

for n in target_maturities:
    current_yield = liu_wu[f'{12*n}_m']
    future_previous_yield = liu_wu[
        f'{12*(n-1)}_m'
    ].shift(-12)
    short_rate = liu_wu['12_m']

    excess_returns[f'xr_{n}y'] = (
        -(n - 1) * (future_previous_yield - current_yield) + (current_yield - short_rate)
    )

excess_returns = excess_returns.dropna()

Processing FRED-MD transformation codes


In [31]:
fred_md['date'] = pd.PeriodIndex(
    pd.to_datetime(fred_md['sasdate']),
    freq='M'
)

fred_md = fred_md.drop(
    columns='sasdate'
).set_index('date')


def transform_series(series, code):
    series = pd.to_numeric(series, errors='coerce')

    if code == 1:
        return series

    elif code == 2:
        return series.diff()

    elif code == 3:
        return series.diff().diff()

    elif code == 4:
        return np.log(
            series.where(series > 1e-6)
        )

    elif code == 5:
        log_series = np.log(
            series.where(series > 1e-6)
        )
        return log_series.diff()

    elif code == 6:
        log_series = np.log(
            series.where(series > 1e-6)
        )
        return log_series.diff().diff()

    elif code == 7:
        percent_change = series.pct_change()
        return percent_change.diff()

    else:
        return series * np.nan


transformed_data = {}

for column in fred_md.columns:
    code = int(float(fred_transform_codes[column]))

    transformed_data[column] = transform_series(
        fred_md[column],
        code
    )

macro_panel = pd.DataFrame(
    transformed_data,
    index=fred_md.index
)

final data

In [32]:
X_yield_only = forward_rates.copy()
X_macro_yield = forward_rates.join(
    macro_panel,
    how='inner'
)
tree_dataset = X_macro_yield.join(
    excess_returns,
    how='inner'
)

In [33]:
processed_dir.mkdir(parents=True, exist_ok=True)

forward_rates.reset_index().to_csv(
    processed_dir / 'forward_rates.csv',
    index=False
)

excess_returns.reset_index().to_csv(
    processed_dir / 'excess_returns.csv',
    index=False
)

macro_panel.reset_index().to_csv(
    processed_dir / 'macro_panel.csv',
    index=False
)

X_yield_only.reset_index().to_csv(
    processed_dir / 'X_yield_only.csv',
    index=False
)

X_macro_yield.reset_index().to_csv(
    processed_dir / 'X_macro_yield.csv',
    index=False
)

tree_dataset.reset_index().to_csv(
    processed_dir / 'tree_dataset.csv',
    index=False
)

liu_wu — no target — original Liu–Wu yields

forward_rates — no target — short_rate, fwd_2y–fwd_10y

excess_returns — with target — xr_2y–xr_10y

fred_md — no target — original FRED-MD variables

macro_panel — no target — transformed macro variables

X_yield_only — no target — copy of forward_rates

X_macro_yield — no target — forward rates + macro variables

tree_dataset — with target — forward rates + macro variables + xr_*
